In [ ]:
# ============================================================
#    Grayscale Image Decryption using Cross-Coupled PWLCM
#
#   NOTE: You must enter the SAME keys and SHA-256 hash
#         that were used during encryption.
# ============================================================

import hashlib
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files


# ─────────────────────────────────────────────
# STEP 1: Upload encrypted image
# ─────────────────────────────────────────────
print("Upload the encrypted image:")
uploaded = files.upload()
image_filename = list(uploaded.keys())[0]

cipher = np.array(Image.open(image_filename).convert('L'), dtype=np.uint8)
M, N   = cipher.shape
print(f"Encrypted image loaded: {image_filename}  |  Size: {N}x{M} px")


# ─────────────────────────────────────────────
# STEP 2: Enter the original keys used during encryption
#         (copy these from the encryption output)
# ─────────────────────────────────────────────
xm  = 0.25686446985353   # <-- change if you used different keys
xx1 = 0.35488659076447
ym  = 0.26457834689785
yx1 = 0.36789543267894

# SHA-256 hash of the ORIGINAL plain image (printed during encryption)
sha_hash = input("\nPaste the SHA-256 hash from encryption output: ").strip()


# ─────────────────────────────────────────────
# PWLCM: Piece-wise Linear Chaotic Map
# ─────────────────────────────────────────────
def pwlcm(s, a):
    if 0 <= s < a:
        return s / a
    elif a <= s < 0.5:
        return (s - a) / (0.5 - a)
    else:
        return 1.0 - s


# ─────────────────────────────────────────────
# REGENERATE KEYS FROM SHA-256 HASH
# ─────────────────────────────────────────────
def regenerate_keys(sha_hash, xm, xx1, ym, yx1):
    hd  = [int(sha_hash[i], 16) for i in range(64)]

    def _key(original, digits):
        s = sum(digits)
        return original - math.floor(s / 1e15) - math.ceil(s / 1e15) * 0.01

    eps = 1e-10
    x1 = np.clip(abs(_key(xm,  hd[0:16])),  eps, 1 - eps)
    ux = np.clip(abs(_key(xx1, hd[16:32])), eps, 0.5 - eps)
    y1 = np.clip(abs(_key(ym,  hd[32:48])), eps, 1 - eps)
    yx = np.clip(abs(_key(yx1, hd[48:64])), eps, 0.5 - eps)

    return x1, ux, y1, yx


# ─────────────────────────────────────────────
# CROSS-COUPLED SEQUENCE GENERATION
# ─────────────────────────────────────────────
def cross_coupled_sequences(x1, ux, y1, yx, Mx):
    x_seq = np.zeros(Mx)
    y_seq = np.zeros(Mx)
    x_seq[0], y_seq[0] = x1, y1
    x_seq[1] = pwlcm(x1, ux)
    y_seq[1] = pwlcm(y1, yx)

    for i in range(1, Mx - 1):
        yi = y_seq[i] if y_seq[i] <= 0.5 else 1.0 - y_seq[i]
        x_seq[i + 1] = pwlcm(yi, ux)
        xi = x_seq[i + 1] if x_seq[i + 1] <= 0.5 else 1.0 - x_seq[i + 1]
        y_seq[i + 1] = pwlcm(xi, yx)

    return x_seq, y_seq


# ─────────────────────────────────────────────
# DECRYPTION
# ─────────────────────────────────────────────
def decrypt(cipher, sha_hash, xm, xx1, ym, yx1):
    M, N = cipher.shape
    Mx   = max(M, N)

    # Regenerate keys
    x1, ux, y1, yx = regenerate_keys(sha_hash, xm, xx1, ym, yx1)
    print(f"\nRegenerated keys:")
    print(f"  x(1) = {x1:.15f}   ux = {ux:.15f}")
    print(f"  y(1) = {y1:.15f}   yx = {yx:.15f}")

    # Cross-coupled sequences
    x_seq, y_seq = cross_coupled_sequences(x1, ux, y1, yx, Mx)

    # Permutation indices (same as encryption)
    row_index = np.argsort(x_seq[(Mx - M):])
    col_index = np.argsort(y_seq[(Mx - N):])

    # Key sequences for diffusion
    x_raw = (np.round(x_seq[(Mx - M):] * 1e6) % 256).astype(np.uint8)
    y_raw = (np.round(y_seq[(Mx - N):] * 1e6) % 256).astype(np.uint8)
    x_key = np.tile(x_raw, math.ceil(N / M))[:N]   # length N
    y_key = np.tile(y_raw, math.ceil(M / N))[:M]   # length M

    # ── Inverse Column Diffusion ──
    # First column XORed with y_key; remaining columns XORed with previous cipher column
    col_undiff = np.zeros_like(cipher)
    col_undiff[:, 0] = cipher[:, 0] ^ y_key
    for j in range(1, N):
        col_undiff[:, j] = cipher[:, j] ^ cipher[:, j - 1]

    # ── Inverse Row Diffusion ──
    # First row XORed with x_key; remaining rows XORed with previous cipher row
    row_undiff = np.zeros_like(col_undiff)
    row_undiff[0] = col_undiff[0] ^ x_key
    for i in range(1, M):
        row_undiff[i] = col_undiff[i] ^ col_undiff[i - 1]

    # ── Inverse Permutation (de-shuffle columns then rows) ──
    inv_col = np.argsort(col_index)
    inv_row = np.argsort(row_index)
    decrypted = row_undiff[:, inv_col][inv_row, :]

    return decrypted


# ─────────────────────────────────────────────
# STEP 3: Run Decryption
# ─────────────────────────────────────────────
decrypted = decrypt(cipher, sha_hash, xm, xx1, ym, yx1)
print("\nDecryption complete!")


# ─────────────────────────────────────────────
# STEP 4: Display encrypted and decrypted images
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cipher, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Encrypted Image", fontsize=14)
axes[0].axis('off')

axes[1].imshow(decrypted, cmap='gray', vmin=0, vmax=255)
axes[1].set_title("Decrypted Image", fontsize=14)
axes[1].axis('off')

plt.suptitle("Cross-Coupled PWLCM Image Decryption", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("decryption_result.png", dpi=150, bbox_inches='tight')
plt.show()
print("Side-by-side comparison saved as: decryption_result.png")


# ─────────────────────────────────────────────
# STEP 5: Save and download decrypted image
# ─────────────────────────────────────────────
dec_filename = "decrypted_" + image_filename.rsplit('.', 1)[0] + ".png"
Image.fromarray(decrypted).save(dec_filename)
print(f"\nDecrypted image saved as: {dec_filename}")

files.download(dec_filename)
files.download("decryption_result.png")